In [1]:
!pip install datasets


In [2]:
from datasets import load_dataset

dataset = load_dataset("hibana2077/UCF-Crime-Dataset", split="train")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ucf-crime-dataset.zip:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

# Task
The task is to build a video classification model for thief detection using the "hibana2077/UCF-Crime-Dataset" dataset. This involves analyzing the dataset structure, defining video preprocessing functions, implementing a custom video dataset and data loader, selecting and initializing a suitable video classification model, configuring training parameters, training the model, evaluating its performance, optimizing it through hyperparameter tuning, and demonstrating inference on a new video with visualization of the prediction result.

## Analyze Dataset Structure

### Subtask:
Examine the loaded `dataset` object to understand its features, such as video paths, labels, and any other relevant metadata, to inform subsequent processing steps.


**Reasoning**:
To understand the dataset's structure, I will print the dataset object, its features, and a single example to see the actual data, including video paths and labels.



In [3]:
print("Dataset object overview:")
print(dataset)

print("\nDataset features:")
print(dataset.features)

print("\nFirst example from the dataset:")
print(dataset[0])

Dataset object overview:
Dataset({
    features: ['image', 'label'],
    num_rows: 1377653
})

Dataset features:
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism'])}

First example from the dataset:
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=64x64 at 0x7BBA9F072E40>, 'label': 0}


## Define Video Preprocessing Functions

### Subtask:
Create Python functions to handle essential video preprocessing tasks, including extracting frames, resizing them to a consistent dimension, and normalizing pixel values for neural network input.


**Reasoning**:
First, I will import the necessary libraries, `torch` and `torchvision.transforms`, which are essential for defining the image preprocessing pipeline.



In [4]:
import torch
import torchvision.transforms as transforms
from PIL import Image

print("Libraries imported: torch, torchvision.transforms, PIL")

Libraries imported: torch, torchvision.transforms, PIL


**Reasoning**:
I will define the `preprocess_frame` function to resize, convert to tensor, and normalize a single PIL Image, and then define an `extract_frames` function that applies this preprocessing to a list of PIL Images to simulate processing frames from a video, as per the subtask instructions.



In [5]:
def preprocess_frame(frame: Image.Image) -> torch.Tensor:
    """Preprocesses a single PIL Image frame for neural network input."""
    transform = transforms.Compose([
        transforms.Resize((64, 64)),  # Resize to 64x64, consistent with dataset
        transforms.ToTensor(),        # Convert PIL Image to PyTorch tensor (values in [0, 1])
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet normalization
    ])
    return transform(frame)

def extract_frames(image_list: list[Image.Image]) -> torch.Tensor:
    """Applies preprocessing to a list of PIL Image frames and stacks them into a tensor."""
    preprocessed_frames = [preprocess_frame(frame) for frame in image_list]
    return torch.stack(preprocessed_frames) # Stack into a single tensor (T, C, H, W)

print("Preprocessing functions `preprocess_frame` and `extract_frames` defined.")


Preprocessing functions `preprocess_frame` and `extract_frames` defined.


## Implement Custom Video Dataset and DataLoader

### Subtask:
Develop a custom `torch.utils.data.Dataset` or equivalent for TensorFlow to efficiently load video frames and their corresponding labels, and set up `DataLoader`s for batching during training and validation.


**Reasoning**:
I need to import the necessary `Dataset` and `DataLoader` classes from `torch.utils.data` as the first step towards implementing the custom video dataset and data loader.



In [6]:
from torch.utils.data import Dataset, DataLoader

print("Imported Dataset and DataLoader from torch.utils.data.")

Imported Dataset and DataLoader from torch.utils.data.


**Reasoning**:
I will define the `VideoDataset` class as instructed, including `__init__`, `__len__`, and `__getitem__` methods, and then instantiate `VideoDataset` and `DataLoader` objects.



In [7]:
class VideoDataset(Dataset):
    def __init__(self, dataset, num_frames: int, transform):
        self.dataset = dataset
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # Retrieve the single image and label for the given index
        item = self.dataset[idx]
        image_pil = item['image']
        label = item['label']

        # Simulate a video by repeating the single image num_frames times
        # This assumes the dataset provides a single image per entry and we need to simulate a sequence
        simulated_frames = [image_pil] * self.num_frames

        # Apply the transformation (extract_frames) to the list of simulated frames
        # self.transform here is expected to be the extract_frames function
        preprocessed_frames = self.transform(simulated_frames)

        return preprocessed_frames, label

# Instantiate the custom dataset
num_frames_per_video = 16 # Example: each 'video' will consist of 16 frames

# The `extract_frames` function defined previously is the transform needed
video_dataset = VideoDataset(dataset=dataset, num_frames=num_frames_per_video, transform=extract_frames)

# Set up DataLoader for batching
batch_size = 32
shuffle = True
num_workers = 2 # Using 2 workers for loading data

video_dataloader = DataLoader(video_dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)

print(f"VideoDataset created with {len(video_dataset)} samples.")
print(f"DataLoader created with batch_size={batch_size}, shuffle={shuffle}, num_workers={num_workers}.")

# Verify one batch
for frames, labels in video_dataloader:
    print(f"Shape of frames in a batch: {frames.shape}") # Expected: (batch_size, num_frames, C, H, W)
    print(f"Shape of labels in a batch: {labels.shape}") # Expected: (batch_size,)
    break # Just take one batch to verify

VideoDataset created with 1377653 samples.
DataLoader created with batch_size=32, shuffle=True, num_workers=2.
Shape of frames in a batch: torch.Size([32, 16, 3, 64, 64])
Shape of labels in a batch: torch.Size([32])


## Select and Initialize Video Classification Model

### Subtask:
Choose and initialize an appropriate deep learning model architecture for video classification.


**Reasoning**:
As instructed, I will import `torch.nn` as `nn`, define a custom `VideoClassifier` class with 3D convolutional layers, pooling, and fully connected layers, including a final output layer for 14 classes. I will also implement the `forward` pass with appropriate tensor permutation, then instantiate and print the model to verify its structure.



In [8]:
import torch.nn as nn

class VideoClassifier(nn.Module):
    def __init__(self, num_classes=14):
        super(VideoClassifier, self).__init__()
        # Define 3D convolutional layers
        self.conv_layers = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)), # Pool spatially, keep temporal
            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))  # Pool temporally and spatially
        )

        # Determine input features for the fully connected layers
        # Assuming input shape (batch_size, num_frames, C, H, W) -> (batch_size, C, num_frames, H, W)
        # After first conv/pool: (B, 32, T, H/2, W/2)
        # After second conv/pool: (B, 64, T/2, H/4, W/4)
        # With H=64, W=64, T=16:
        # After first pool: (B, 32, 16, 32, 32)
        # After second pool: (B, 64, 8, 16, 16)
        # Flattened size: 64 * 8 * 16 * 16 = 131072

        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 8 * 16 * 16, 512), # Adjust based on actual output of conv layers
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # x shape: (batch_size, num_frames, C, H, W)
        # Permute to (batch_size, C, num_frames, H, W) for 3D convolutions
        x = x.permute(0, 2, 1, 3, 4)

        x = self.conv_layers(x)

        # Flatten the output for fully connected layers
        x = x.view(x.size(0), -1)

        x = self.fc_layers(x)
        return x

# Instantiate the model
model = VideoClassifier(num_classes=len(dataset.features['label'].names))

print("Video Classification Model Initialized.")
print("Model Architecture:")
print(model)


Video Classification Model Initialized.
Model Architecture:
VideoClassifier(
  (conv_layers): Sequential(
    (0): Conv3d(3, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): ReLU()
    (2): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
    (3): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): ReLU()
    (5): MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=131072, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=512, out_features=14, bias=True)
  )
)


## Configure Training Parameters

### Subtask:
Set up the training configuration, including defining the loss function (e.g., Cross-Entropy), selecting an optimizer (e.g., Adam), and specifying learning rate schedules and other hyperparameters.


**Reasoning**:
As instructed, I will define the loss function, select an optimizer, determine the device for training, and move the model to that device to set up the training configuration.



In [9]:
import torch.optim as optim

# 1. Define the loss function
criterion = nn.CrossEntropyLoss()

# 2. Select an optimizer
learning_rate = 1e-3
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 3. Define the device for training (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 4. Move the model to the selected device
model.to(device)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")
print(f"Training device: {device}")
print("Model moved to the selected device.")

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Training device: cpu
Model moved to the selected device.


## Train the Model

### Subtask:
Implement and execute the training loop for the selected model over multiple epochs, performing forward and backward passes, calculating loss, and updating model weights using the chosen optimizer.


### Implement and Execute the Training Loop

**Reasoning**:
Now, I will implement the core training loop. This involves iterating through a specified number of epochs, and within each epoch, processing batches of data from the `video_dataloader`. For each batch, I will perform the following steps:

1.  **Move Data to Device**: Transfer the input frames and corresponding labels to the designated training device (GPU if available, otherwise CPU).
2.  **Zero Gradients**: Clear any previously accumulated gradients to prevent them from interfering with the current backpropagation step.
3.  **Forward Pass**: Feed the preprocessed frames through the `VideoClassifier` model to obtain predictions.
4.  **Calculate Loss**: Compute the difference between the model's predictions and the true labels using the `CrossEntropyLoss` criterion.
5.  **Backward Pass**: Calculate gradients of the loss with respect to the model's parameters.
6.  **Optimizer Step**: Update the model's weights based on the calculated gradients and the learning rate configured in the `Adam` optimizer.
7.  **Monitor Progress**: Track and print the average loss for each epoch to observe the model's learning progress.

**Reasoning**:
I will now implement the training loop according to the specified instructions, including iterating through epochs, processing batches, moving data to the device, performing forward and backward passes, calculating loss, and updating model weights.



In [ ]:
num_epochs = 3 # Set the number of training epochs

for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    for i, (frames, labels) in enumerate(video_dataloader):
        # 4. Move the frames and labels to the device
        frames = frames.to(device)
        labels = labels.to(device)

        # 5. Zero out the gradients
        optimizer.zero_grad()

        # 6. Perform a forward pass
        outputs = model(frames)

        # 7. Calculate the loss
        loss = criterion(outputs, labels)

        # 8. Perform a backward pass
        loss.backward()

        # 9. Update the model's weights
        optimizer.step()

        running_loss += loss.item()

        # Optional: Print loss at intervals within an epoch
        if i % 100 == 99:    # print every 100 mini-batches
            print(f"Epoch [{epoch + 1}/{num_epochs}], Batch [{i + 1}/{len(video_dataloader)}], Loss: {running_loss / 100:.4f}")
            running_loss = 0.0

    # 10. Print the average loss for the epoch
    print(f"Epoch {epoch+1} finished. Average Loss: {running_loss / len(video_dataloader):.4f}")

print("Training complete.")